# 14. Container Deployment and Monitoring

This notebook documents and demonstrates how to deploy the SupportAI service using Docker and Docker Compose, and how to verify the integrated monitoring setup with Prometheus and Grafana.

## 1. Docker Build Instructions

The application contains a `Dockerfile` at the root of the repository. To build the production Docker image, run:

```bash
docker build -t supportai-api:latest .
```

## 2. Multi-Service Container Orchestration

A `docker-compose.yml` file is provided to link three essential services:
1. **FastAPI Web Service** (`web`): Serves the REST API on port `8000`.
2. **Prometheus Server** (`prometheus`): Scrapes telemetry metrics from `http://web:8000/metrics` every 15s, serving on port `9090`.
3. **Grafana Dashboard** (`grafana`): Connects to Prometheus as a data source to visualize request rates, latency, and prediction calibration on port `3000`.

To start all services, run:
```bash
docker-compose up --build -d
```

## 3. Configuration Files Overview

Let's inspect the Prometheus scraping configuration file [`configs/prometheus.yml`](../configs/prometheus.yml) and the Docker Compose file [`docker-compose.yml`](../docker-compose.yml).

In [1]:
print("=== configs/prometheus.yml ===\n")
with open("../configs/prometheus.yml", "r") as f:
    print(f.read())

print("\n=== docker-compose.yml ===\n")
with open("../docker-compose.yml", "r") as f:
    print(f.read())

=== configs/prometheus.yml ===

global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  - job_name: 'supportai-api'
    scrape_interval: 5s
    static_configs:
      - targets: ['web:8000']


=== docker-compose.yml ===

version: '3.8'

services:
  web:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "8000:8000"
    environment:
      - TESTING=false
      - LOG_LEVEL=INFO
    volumes:
      - ./logs:/app/logs
      - ./outputs:/app/outputs
      - ./mlruns:/app/mlruns
    restart: unless-stopped

  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./configs/prometheus.yml:/etc/prometheus/prometheus.yml:ro
    command:
      - "--config.file=/etc/prometheus/prometheus.yml"
      - "--storage.tsdb.path=/prometheus"
    restart: unless-stopped
    depends_on:
      - web

  grafana:
    image: grafana/grafana:latest
    ports:
      - "3000:3000"
    environment:
      - GF_SECURITY_ADMIN_PASS

## 4. Checking Structured Trace Logs

The FastAPI application writes structured JSON logs for requests (excluding health/metrics checks) to `logs/traces.jsonl`. Let's print out the latest log entry to verify the schema structure:

In [2]:
import json
from pathlib import Path

traces_file = Path("../logs/traces.jsonl")
if traces_file.exists():
    with open(traces_file, "r") as f:
        lines = f.readlines()
        if lines:
            print(json.dumps(json.loads(lines[-1]), indent=2))
        else:
            print("traces.jsonl is empty")
else:
    print("No traces.jsonl log file found. Run a prediction first!")

{
  "request_id": "e0dcf076-5083-492a-b07f-6f2e1da6c791",
  "timestamp": "2026-07-13T16:24:27Z",
  "endpoint": "/explain",
  "method": "POST",
  "status_code": 200,
  "latency_seconds": 0.08841347694396973,
  "version": "0.1.0",
  "git_commit": "702a29409e72c12741ec4dc8c30ac128501307d4",
  "experiment_id": "default"
}
